# Global EV Charging Stations Analysis

## Project Objective

The objective of this project is to analyze global electric vehicle (EV) charging station data to uncover patterns in charging infrastructure across different countries, cities, operators, and connector types. The analysis aims to identify trends in EV charging availability, evaluate the distribution of charging stations, and generate insights that can support infrastructure planning and the continued adoption of electric vehicles.

In [5]:
import pandas as pd
import numpy as np

### Data Inspection

In [6]:
df = pd.read_csv("ev_stations_2025.csv")

In [ ]:
df.head()

,id,title,address,town,state,postcode,country,lat,lon,operator,status,num_connectors,connector_types,date_added
0,462769,Electra - Wambrechies - Volfoni,81 Av. Clément Ader,Wambrechies,NaN,59118,FR,50.685653,3.062410,Electra,Operational,2,CCS (Type 2)|Type 2 (Socket Only),2025-11-02 09:58:00+00:00
1,462768,Electra - Tourcoing - Action,185 rue du Touquet,Tourcoing,NaN,59200,FR,50.723171,3.180636,Electra,Operational,3,CCS (Type 2)|CHAdeMO|Type 2 (Socket Only),2025-11-02 09:55:00+00:00
2,462767,Electra - Bondues - Sure Hotel by Best Western...,3 Av. Henri Becquerel,Bondues,NaN,59910,FR,50.722535,3.129496,Electra,Operational,2,CCS (Type 2)|Type 2 (Socket Only),2025-11-02 09:50:00+00:00
3,462766,Electra - Bousbecque - Intermarché,Rue Auger,Bousbecque,NaN,59166,FR,50.770139,3.083454,Electra,Operational,2,CCS (Type 2)|Type 2 (Socket Only),2025-11-02 09:46:00+00:00
4,462765,Electra - Halluin - Intermarché,Boulevard de Roncq,Halluin,NaN,59250,FR,50.769938,3.124030,Electra,Operational,2,CCS (Type 2)|Type 2 (Socket Only),2025-11-02 09:42:00+00:00


In [5]:
df.shape

(10000, 14)

In [6]:
df.columns

Index(['id', 'title', 'address', 'town', 'state', 'postcode', 'country', 'lat',
       'lon', 'operator', 'status', 'num_connectors', 'connector_types',
       'date_added'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               10000 non-null  int64  
 1   title            10000 non-null  object 
 2   address          9999 non-null   object 
 3   town             9797 non-null   object 
 4   state            7131 non-null   object 
 5   postcode         8330 non-null   object 
 6   country          10000 non-null  object 
 7   lat              10000 non-null  float64
 8   lon              10000 non-null  float64
 9   operator         9402 non-null   object 
 10  status           10000 non-null  object 
 11  num_connectors   10000 non-null  int64  
 12  connector_types  9995 non-null   object 
 13  date_added       10000 non-null  object 
dtypes: float64(2), int64(2), object(10)
memory usage: 1.1+ MB


In [8]:
df.describe()

,id,lat,lon,num_connectors
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,416572.240600,42.425898,-52.399215,1.456500
std,40129.224879,14.651593,63.942829,0.989851
min,374562.000000,-43.904364,-159.356940,0.000000
25%,378741.750000,42.017013,-112.697935,1.000000
50%,384381.500000,45.778014,-73.441045,1.000000
75%,459627.250000,48.673134,-0.361442,2.000000
max,462769.000000,68.377358,174.756158,26.000000


In [54]:
df.isnull().sum()

id                    0
title                 0
address               1
town                203
state              2869
postcode           1670
country               0
lat                   0
lon                   0
operator            598
status                0
num_connectors        0
connector_types       5
date_added            0
dtype: int64

## Data Cleaning

Data cleaning is an essential step before analysis. In this section, the dataset is examined for data quality issues such as duplicate records, missing values, inconsistent entries, and incorrect data types. Appropriate cleaning techniques are then applied to improve the accuracy and consistency of the analysis.

In [7]:
#convert our original df into a new copy as clean_df variable for cleansing
clean_df = df.copy()

In [53]:
#tells you how many missing values there are in each column?
clean_df.isnull().sum()

id                    0
title                 0
address               0
town                203
state              2869
postcode           1670
country               0
lat                   0
lon                   0
operator              0
status                0
num_connectors        0
connector_types       5
date_added            0
dtype: int64

In [ ]:
# tells you where the missing values for address column?
clean_df[clean_df["address"].isnull()]

In [ ]:
# tells you where the missing values for connector_types column?
clean_df[clean_df["connector_types"].isnull()]

In [ ]:
# tells you where the missing values for town column ?
clean_df[clean_df["town"].isnull()]

In [ ]:
# tells you where the missing values for state column ?
clean_df[clean_df["state"].isnull()]

In [ ]:
# tells you where the missing values for operator column ?
clean_df[clean_df["operator"].isnull()]

In [ ]:
# tells you where the missing values for country column ?
clean_df[clean_df["country"].isnull()]

##### Handling Missing Values

Before replacing missing values, each affected record was inspected individually. Where the missing information could be confidently inferred from other columns or publicly available company information, the missing value was filled. Otherwise, the missing value was retained to avoid introducing unsupported assumptions into the dataset.

In [26]:
# clean the address column where missing values by filling it through the condition of

# For rows where 'address' is missing, copy the value from the 'title' column and give it same value found there.
clean_df.loc[
    clean_df["address"].isnull(),"address"] = clean_df.loc[ clean_df["address"].isnull(),"title"
]

#verify if it has been applied
clean_df["address"].isnull().sum()

np.int64(0)

In [27]:
#clean the operator column where missing values by filling it "unknown Operator" 

clean_df["operator"] = clean_df["operator"].fillna("(Unknown Operator)")

#verify if it has been applied
clean_df["operator"].isnull().sum()

np.int64(0)

In [25]:
#here we created a dic variable country_mapping to store our full country name using each country code as their full country name
country_mapping = {
    "AE": "United Arab Emirates",
    "AL": "Albania",
    "AU": "Australia",
    "BA": "Bosnia and Herzegovina",
    "BG": "Bulgaria",
    "CA": "Canada",
    "CH": "Switzerland",
    "CN": "China",
    "CO": "Colombia",
    "CR": "Costa Rica",
    "CY": "Cyprus",
    "CZ": "Czech Republic",
    "DE": "Germany",
    "DK": "Denmark",
    "DO": "Dominican Republic",
    "EC": "Ecuador",
    "EE": "Estonia",
    "EG": "Egypt",
    "GB": "United Kingdom",
    "GE": "Georgia",
    "GR": "Greece",
    "GT": "Guatemala",
    "HR": "Croatia",
    "ID": "Indonesia",
    "IE": "Ireland",
    "IL": "Israel",
    "IN": "India",
    "JM": "Jamaica",
    "JO": "Jordan",
    "JP": "Japan",
    "LI": "Liechtenstein",
    "LK": "Sri Lanka",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "MA": "Morocco",
    "MD": "Moldova",
    "MT": "Malta",
    "MX": "Mexico",
    "MY": "Malaysia",
    "NO": "Norway",
    "NP": "Nepal",
    "NZ": "New Zealand",
    "PA": "Panama",
    "PT": "Portugal",
    "PY": "Paraguay",
    "RO": "Romania",
    "RU": "Russia",
    "SE": "Sweden",
    "SI": "Slovenia",
    "SK": "Slovakia",
    "UA": "Ukraine",
    "UY": "Uruguay",
    "UZ": "Uzbekistan",
    "VN": "Vietnam",
    "ZA": "South Africa"
}

#The entire value inside the column country to be replaced using what we store in country_mapping & apply it back in the df clean_df["country"]
clean_df["country"] = clean_df["country"].replace(country_mapping)

#show list of country values and sort it out without any duplicate
clean_df["country"].sort_values().unique()

0              France
1              France
2              France
3              France
4              France
            ...      
9995    United States
9996           Canada
9997           Canada
9998    United States
9999           Canada
Name: country, Length: 10000, dtype: object

In [8]:
#changing the date from default as string to real pandas datetime for easier analysis
clean_df["date_added"] = pd.to_datetime(clean_df["date_added"])

#verify if it has been converted (the Dtype must show datetime)
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   id               10000 non-null  int64              
 1   title            10000 non-null  object             
 2   address          9999 non-null   object             
 3   town             9797 non-null   object             
 4   state            7131 non-null   object             
 5   postcode         8330 non-null   object             
 6   country          10000 non-null  object             
 7   lat              10000 non-null  float64            
 8   lon              10000 non-null  float64            
 9   operator         9402 non-null   object             
 10  status           10000 non-null  object             
 11  num_connectors   10000 non-null  int64              
 12  connector_types  9995 non-null   object             
 13  date_added       

In [29]:
#the cleaned dataset  should be exported & saved, no index.
clean_df.to_csv("cleaned_ev_stations.csv", index=False)

# Exploratory Data Analysis (EDA)

This section explores the EV charging station dataset to identify patterns, trends, and insights related to charging infrastructure, geographic distribution, charging operators, connector availability, and station characteristics.

#### Question 1

Which country has the highest number of EV charging stations?

In [45]:
clean_df["country"].value_counts()

country
Canada                    4135
United States             2266
Spain                      888
Russia                     392
Malaysia                   377
                          ... 
China                        1
Jamaica                      1
Malta                        1
Bosnia and Herzegovina       1
Norway                       1
Name: count, Length: 72, dtype: int64

### Insight

Canada has the highest number of EV charging stations in the dataset with **4,135** stations, followed by the **United States (2,266)** and **Spain (888)**. The large gap between Canada and the remaining countries suggests that Canada has the most extensive EV charging station coverage within this dataset.

#### Question 2

Which operators own the largest charging networks?

In [51]:
clean_df.groupby("operator")["num_connectors"].sum().sort_values(ascending=False)

operator
Circuit Electrique             2308
ChargePoint                    1410
(Unknown Operator)             1298
PowerGo                        1290
ChargeSini                     1125
                               ... 
TurnOnGreen                       1
Go Electric E-Mobility (BR)       1
GioEV (TR)                        1
Volta Charging                    1
Evmapa (CZ)                       1
Name: num_connectors, Length: 263, dtype: int64

### Insight

Circuit Electrique operates the largest charging network in the dataset with **2,308** charging connectors, followed by **ChargePoint (1,410)** and **PowerGo (1,290)**. The presence of **(Unknown Operator)** among the top results indicates that some operator information is missing, which may affect the completeness of operator-level analysis.

#### Question 3

Which connector type configurations are most commonly available across EV charging stations?

In [56]:
clean_df["connector_types"].value_counts()

connector_types
Type 1 (J1772)                                                                                                                                   4418
Type 2 (Socket Only)                                                                                                                              806
CCS (Type 2)                                                                                                                                      722
CHAdeMO|CCS (Type 1)                                                                                                                              714
CCS (Type 1)                                                                                                                                      490
                                                                                                                                                 ... 
CCS (Type 2)|CHAdeMO|CCS (Type 2)                                                   

### Insight

The most common connector configuration in the dataset is **Type 1 (J1772)**, appearing at **4,418** charging stations. Other frequently available configurations include **Type 2 (Socket Only)**, **CCS (Type 2)**, **CHAdeMO|CCS (Type 1)**, and **CCS (Type 1)**. This suggests that a few connector standards dominate the charging infrastructure, while many other configurations are relatively uncommon.

#### Question 4

Which cities have the most charging stations?

In [62]:
clean_df["town"].value_counts()

town
Montréal            686
Seattle             202
Vancouver           152
Portland            109
Québec               96
                   ... 
Jalan Duta            1
Ampang                1
Tropicana Indah       1
Jalan Tun Ismail      1
HAPPY VALLEY          1
Name: count, Length: 3852, dtype: int64

### Insight

Montréal has the highest number of EV charging stations in the dataset with **686** stations, significantly more than **Seattle (202)**, **Vancouver (152)**, **Portland (109)**, and **Québec (96)**. This suggests that Montréal has the most extensive charging station coverage among the cities represented in the dataset, indicating stronger EV charging infrastructure in that location.

#### Question 5

What is the operational status of charging stations?

In [64]:
clean_df["status"].value_counts()

status
Operational                   9301
Planned For Future Date        302
Temporarily Unavailable        176
Not Operational                136
Unknown                         80
Partly Operational (Mixed)       5
Name: count, dtype: int64

### Insight

The vast majority of EV charging stations in the dataset are **operational**, with **9,301** active stations. Only a small number are **planned (302)**, **temporarily unavailable (176)**, or **not operational (136)**, while **80** stations have an **unknown** status and **5** are **partly operational**. This suggests that most charging infrastructure in the dataset is currently available for public use, indicating a high level of network readiness.

#### Question 6

Which countries have the greatest connector diversity?

In [78]:
clean_df.groupby("country")["connector_types"].nunique().sort_values(ascending=False)

country
Russia           55
Brazil           29
United States    28
Spain            27
Malaysia         25
                 ..
Malta             1
Portugal          1
Moldova           1
Paraguay          1
Albania           1
Name: connector_types, Length: 72, dtype: int64

### Insight

Russia has the greatest diversity of connector configurations in the dataset, with **55** unique connector configurations, followed by **Brazil (29)**, **United States (28)**, **Spain (27)**, and **Malaysia (25)**. This suggests that these countries support a wider variety of charging connector configurations, potentially improving compatibility with different EV models and charging requirements.

#### Question 7

Which countries have the highest average number of connectors per charging station?

In [79]:
clean_df.groupby("country")["num_connectors"].mean().sort_values(ascending=False)

country
Norway                    8.000000
Luxembourg                4.500000
Jordan                    3.625000
Lithuania                 3.476190
Netherlands               3.242105
                            ...   
Bulgaria                  1.000000
Bosnia and Herzegovina    1.000000
Vietnam                   1.000000
Dominican Republic        0.857143
Nepal                     0.500000
Name: num_connectors, Length: 72, dtype: float64

In [80]:
clean_df["country"].value_counts().loc[["Norway", "Luxembourg", "Jordan", "Lithuania", "Netherlands"]]

country
Norway          1
Luxembourg      2
Jordan          8
Lithuania      21
Netherlands    95
Name: count, dtype: int64

### Insight

Norway has the highest average number of connectors per charging station in the dataset, with an average of **8 connectors** per station. It is followed by **Luxembourg (4.5)**, **Jordan (3.63)**, **Lithuania (3.48)**, and **the Netherlands (3.24)**. This suggests that charging stations in these countries tend to have greater charging capacity, allowing more electric vehicles to charge simultaneously compared to stations in other countries.

#### Question 8

Which operators offer the highest average number of connectors?

In [85]:
clean_df.groupby("operator")["num_connectors"].mean().sort_values(ascending=False)

operator
WeVolt (AU)                    6.000
MER                            6.000
Chargy (LU)                    5.000
Eldrive Lithuania (LT)         4.375
Swisscharge (CH)               4.000
                               ...  
Greenway Polska (PL)           1.000
Powerflex                      1.000
Char.gy                        1.000
Porsche Smart Mobility GmbH    1.000
EVPass (CH)                    1.000
Name: num_connectors, Length: 263, dtype: float64

### Insight

WeVolt (AU) and MER have the highest average number of connectors per charging station, with an average of **6 connectors** each. They are followed by Chargy (LU) with **5 connectors**, Eldrive Lithuania (LT) with **4.38 connectors**, and Swisscharge (CH) with **4 connectors**. This suggests that these operators generally provide charging stations with higher charging capacity compared to many other operators in the dataset.

#### Question 9

Which regions appear underserved?

In [92]:
clean_df.groupby("country")["num_connectors"].sum().sort_values()

country
Albania                      1
Serbia                       1
Portugal                     1
Bosnia and Herzegovina       1
Nepal                        1
                          ... 
Russia                     996
Spain                     1096
Malaysia                  1142
United States             2646
Canada                    5075
Name: num_connectors, Length: 72, dtype: int64

### Insight

Albania, Serbia, Portugal, Bosnia and Herzegovina, and Nepal each have only **1 charging connector** in the dataset, suggesting they appear to have the least charging infrastructure among the countries represented. However, this conclusion is based only on the available dataset and does not necessarily reflect the actual EV charging infrastructure in these countries.

#### Question 10

What are the key characteristics of the global EV charging network?

### Insight

The global EV charging network is unevenly distributed across countries. Canada has the highest number of charging stations, while countries such as Albania, Serbia, Portugal, Bosnia and Herzegovina, and Nepal have very limited charging infrastructure in the dataset. Circuit Electrique operates the largest charging network in terms of total connectors, while operators such as WeVolt (AU) and MER provide the highest average number of connectors per station. Most charging stations are operational, and connector configurations vary across countries, reflecting differences in charging technology and infrastructure development.

## Key Insights

- Canada has the largest number of EV charging stations in the dataset, indicating a well-developed charging network.
- Circuit Electrique operates the largest charging network based on the total number of charging connectors.
- WeVolt (AU) and MER provide the highest average number of connectors per charging station, suggesting greater charging capacity per location.
- Most charging stations in the dataset are operational, indicating strong network availability.
- Connector configurations vary across countries, reflecting differences in charging technologies and infrastructure.
- Some countries, including Albania, Serbia, Portugal, Bosnia and Herzegovina, and Nepal, have very limited charging infrastructure within the dataset, suggesting potential gaps in network coverage.

## Business Recommendations

- Prioritize investment in countries with limited charging infrastructure to improve EV accessibility and encourage adoption.
- Study the deployment strategies of leading operators such as Circuit Electrique to identify best practices that can be replicated in emerging markets.
- Expand the number of connectors at high-demand charging locations to reduce waiting times and improve the user experience.
- Promote standardized connector technologies where possible to improve compatibility across different EV models.
- Continuously monitor charging station status to maintain high network reliability and minimize service disruptions.

## Conclusion

This analysis provides an overview of the global EV charging network by examining station distribution, operator performance, charging capacity, connector diversity, and operational status. The findings show that charging infrastructure is unevenly distributed across countries, with some regions having significantly greater coverage than others. These insights can help policymakers, charging network operators, and investors identify opportunities for infrastructure expansion and make more informed decisions to support the continued growth of electric vehicle adoption.